# 比较分块设置

分块长度和分块方法要分开比较。下面使用当前《南瓜书》、同一组问题和同一个 BM25 检索器重新计算。

表格只回答“需要的页是否进入前 5 条”，不代表模型最终回答正确。

In [1]:
import sys
from pathlib import Path

def find_c7_root(start):
    start = Path(start).resolve()
    for folder in (start, *start.parents):
        if (folder / 'data' / 'dataset/manifest.json').is_file(): return folder
        nested = folder / 'notebook' / 'C7 高级 RAG 技巧'
        if (nested / 'data' / 'dataset/manifest.json').is_file(): return nested
    raise FileNotFoundError('请从仓库根、C7 根或本章目录启动')

course_root = find_c7_root(Path.cwd())
sys.path.insert(0, str(course_root))

from common.eval_utils import (
    build_bm25_chunk_search,
    load_query_catalog,
    load_pdf_pages,
    make_fixed_chunks,
    make_recursive_chunks,
)
from common.nontraining_utils import load_annotation

pages = load_pdf_pages()
case_ids = [
    "model_evaluation_purpose",
    "slater_strong_duality",
    "continuous_attribute_split_points",
    "svm_soft_margin_alpha_bounds",
    "cross_validation_reliability",
    "newton_hessian_cost",
]
case_by_id = {item["id"]: item for item in load_query_catalog()}
cases = [case_by_id[case_id] for case_id in case_ids]

def hit_rate(chunks, top_k=5):
    search = build_bm25_chunk_search(chunks)
    hits = 0
    for case in cases:
        returned_pages = {page for item in search(case["query"], top_k) for page in item.pages}
        annotation = load_annotation(case["id"])
        hits += bool(returned_pages & set(annotation["expected_pages"]))
    return hits / len(cases)

rows = []
for size in (160, 256, 400):
    chunks = make_fixed_chunks(pages, chunk_size=size, overlap=20)
    rows.append((f"定长 {size}/20", len(chunks), hit_rate(chunks)))

sentence_chunks = make_recursive_chunks(pages, chunk_size=256, overlap=20)
rows.append(("句子边界 256/20", len(sentence_chunks), hit_rate(sentence_chunks)))

print("问题数：", len(cases), "；每题取前 5 条")
print("方法                    片段数    命中率")
for name, count, rate in rows:
    print(f"{name:<22}{count:>6}    {rate:.2f}")

问题数： 6 ；每题取前 5 条
方法                    片段数    命中率
定长 160/20               2046    0.83
定长 256/20               1246    0.67
定长 400/20                821    0.83
句子边界 256/20             1363    0.83


## 怎样读这个结果

分块越小，片段数通常越多，建立索引和检索的成本也会改变。命中率相同时，还要看正确内容是否被切断、送入回答的文字量和检索速度。

这里只用 6 个问题保留一个简短的教学对照，不能据此宣布某个长度适用于所有资料。更换资料或向量模型后，需要重新比较。

## 怎样设计一个可解释的分块对照

比较分块设置时，固定文档、问题、向量模型和返回数量，只改变 `chunk_size`、重叠长度或分隔方式，然后计算正确页是否出现在前几条结果中。召回率可以写成：

R = 命中的问题数 / 有效问题总数

这项指标只能回答“证据所在页有没有被找回”，不能证明片段已经包含完整答案，也不能证明最终回答正确。命中率相同时，还要比较片段是否切断句子、上下文字符数、索引大小、检索时间，以及原本正常的问题有没有变差。本页使用六个当前案例重新计算结果。

建议先做小范围筛选，再在固定的验证问题上复查。例如先比较 160、256、400 字符，再比较递归句子边界；若一种方法只提高页命中，却把上下文扩大数倍，应把收益和代价一起记录。


## 页面召回计算的代码写法

下面只展示页面召回计算的代码写法，不产生本页的比较结果；正式比较还要同时检查片段数量、文字量和必要证据覆盖。

```python
def page_recall(search, cases, top_k=5):
    # 只计算必要页是否命中；不要把它当成答案正确率。
    hits = 0
    for case in cases:
        results = search(case["query"], top_k=top_k)
        pages = {page for item in results for page in item.pages}
        hits += bool(pages.intersection(case["expected_pages"]))
    return hits / len(cases)

# 正式比较时还应同时记录：
# 1. 每个结果的字符数和片段数；
# 2. 必要答案要点在返回上下文中的覆盖情况；
# 3. 同一问题在不同模型或不同分块下的首次目标排名。
```
